# Vectro — Cross-Platform Benchmark Results

This notebook loads the JSON records produced by `reproduce_paper.sh` / `reproduce_paper.ps1`
and renders throughput + compression ratio tables for the paper appendix.

Run the benchmarks first:

```bash
make bench-all WAVE=1          # darwin-arm64 + linux-x86_64, 3 runs each
# or
./reproduce_paper.sh --wave 1 --runs 3
```

Then execute this notebook:

```bash
jupyter nbconvert --to pdf notebooks/vectro_paper_results.ipynb --execute
# (this is what `make bench-arxiv` runs)
```

In [ ]:
import json
import statistics
from pathlib import Path

# ── locate repo root regardless of cwd ──────────────────────────────────
NB_DIR = Path(globals()["__vsc_ipynb_file__"]).parent if "__vsc_ipynb_file__" in globals() else Path().resolve()
REPO = NB_DIR.parent if (NB_DIR / "PLAN.md").exists() else NB_DIR
RESULTS = REPO / "results" / "paper"

files = sorted(RESULTS.glob("wave*.json"))
print(f"results dir : {RESULTS}")
print(f"records found: {len(files)}")
for f in files:
    print(f"  {f.name}")

In [ ]:
# ── load + validate ──────────────────────────────────────────────────────
records = []
for fp in files:
    try:
        with open(fp) as fh:
            r = json.load(fh)
        # Accept both the vectro/paper/wave-bench/v2 schema (from
        # reproduce_paper.sh) and the vectro/paper-benchmark/v1 schema
        # (direct invocation of vectro_paper_benchmark.py).
        records.append(r)
    except Exception as exc:
        print(f"WARN: skipping {fp.name}: {exc}")

if not records:
    print("No records found in results/paper/.  Run reproduce_paper.sh first.")
else:
    print(f"Loaded {len(records)} records.")

In [ ]:
# ── throughput summary table ──────────────────────────────────────────────
from collections import defaultdict

# Bucket by (platform, wave, cold/warm)
buckets = defaultdict(list)
for r in records:
    plat = r.get("platform", "?")
    wave = r.get("wave", 0)
    mode = "cold" if r.get("cold") else "warm"
    # throughput is in M vec/s from v2 schema; convert if raw vec/s
    t = r.get("throughput_mean") or r.get("throughput") or 0.0
    buckets[(plat, wave, mode)].append(float(t))

print(f"{'platform':<30} {'wave':>5} {'mode':>6} {'n':>4} {'mean M/s':>9} {'stddev':>8} {'CoV%':>6}")
print("-" * 72)
for (plat, wave, mode), xs in sorted(buckets.items()):
    mean = statistics.mean(xs) if xs else 0.0
    stdev = statistics.pstdev(xs) if len(xs) >= 2 else 0.0
    cov = (100 * stdev / mean) if mean else 0.0
    flag = "  ⚠ CoV>5%" if cov > 5.0 else ""
    print(f"{plat:<30} {wave:>5} {mode:>6} {len(xs):>4} {mean:>9.3f} {stdev:>8.4f} {cov:>6.2f}{flag}")

In [ ]:
# ── matplotlib chart: throughput by platform ──────────────────────────────
try:
    import matplotlib

    matplotlib.use("Agg")  # headless — safe for nbconvert --execute
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    import numpy as np

    plats = sorted({k[0] for k in buckets})
    waves = sorted({k[1] for k in buckets})
    modes = ["warm", "cold"]
    colors = {"warm": "#22d3ee", "cold": "#c084fc"}

    if not plats:
        print("No data to plot.")
    else:
        fig, ax = plt.subplots(figsize=(max(8, len(plats) * 2), 5))
        fig.patch.set_facecolor("#07080d")
        ax.set_facecolor("#0b0d18")
        for spine in ax.spines.values():
            spine.set_color("#1d2235")
        ax.tick_params(colors="#98a0b6", labelsize=8)

        width = 0.35
        x = np.arange(len(plats))
        for wi, wave in enumerate(waves):
            for mi, mode in enumerate(modes):
                means = [statistics.mean(buckets.get((p, wave, mode), [0])) for p in plats]
                offset = (wi - len(waves) / 2 + 0.5) * width + mi * 0.15
                bars = ax.bar(x + offset, means, width * 0.7, label=f"wave {wave} {mode}", color=colors[mode], alpha=0.85 - 0.35 * wi)
                for bar, val in zip(bars, means):
                    if val > 0:
                        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002, f"{val:.2f}", ha="center", va="bottom", fontsize=7, color="#e9ecf6")

        ax.set_xticks(x)
        ax.set_xticklabels([p.replace(" / ", "\n") for p in plats], fontsize=8, color="#e9ecf6")
        ax.set_ylabel("M vec/s (best-of-N INT8 encode)", color="#e9ecf6", fontsize=9)
        ax.set_title("Vectro — INT8 encode throughput by platform", color="#e9ecf6", fontsize=11)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
        ax.legend(fontsize=8, facecolor="#11141f", edgecolor="#1d2235", labelcolor="#e9ecf6", loc="upper left")
        ax.grid(axis="y", color="#1d2235", linewidth=0.5, linestyle="--")

        fig.tight_layout()
        plt.savefig(RESULTS / "throughput_chart.png", dpi=150, bbox_inches="tight", facecolor="#07080d")
        plt.show()
        print("chart saved to results/paper/throughput_chart.png")
except ImportError:
    print("matplotlib not installed — skipping chart (pip install matplotlib)")

In [ ]:
# ── compression ratio table (from vectro_paper_benchmark.py rows) ─────────
ratio_rows = []
for r in records:
    for row in r.get("rows", []):
        ratio_rows.append(
            {
                "table": row.get("table", "?"),
                "n": row.get("n", 0),
                "d": row.get("d", 0),
                "ratio": row.get("ratio", 0.0),
                "cosine": row.get("mean_cosine", 0.0),
                "best_ms": row.get("best_ms", 0.0),
                "M_vec_s": row.get("best_M_vec_s", 0.0),
                "platform": r.get("platform", "?"),
            }
        )

if not ratio_rows:
    print("No per-table rows found (records may use reproduce_paper.sh schema, which omits rows).")
else:
    print(f"{'table':<9} {'n':>8} {'d':>6} {'ratio':>7} {'cosine':>7} {'best ms':>8} {'M/s':>7}  platform")
    print("-" * 80)
    for r in ratio_rows:
        print(f"{r['table']:<9} {r['n']:>8,} {r['d']:>6} {r['ratio']:>6.2f}× {r['cosine']:>7.4f} {r['best_ms']:>8.2f} {r['M_vec_s']:>7.3f}  {r['platform']}")

In [ ]:
# ── SIMD capability summary ───────────────────────────────────────────────
simd_seen = {}
for r in records:
    key = r.get("platform", "?")
    simd_seen.setdefault(key, r.get("simd", "?"))

print("Platform → SIMD path")
print("-" * 50)
for plat, simd in sorted(simd_seen.items()):
    print(f"  {plat:<35}  {simd}")

## Notes

- All throughput figures are **best-of-N** from `reproduce_paper.{sh,ps1}` with `--reps 1 --warmup 0` per call (outer `--runs N` provides the sample set).
- Compression ratios shown are for the INT8 `balanced` profile (≈ 4×), NF4 `quality` profile (≈ 4–8×), and `binary` profile (32×).
- Cosine similarity is the mean over the full n×d batch, reconstructed from quantised codes.
- Platform labels come from Python `platform.system() + platform.machine()`.
- For thermal and CoV metadata, inspect the raw JSON in `results/paper/`.